<a target="_blank" href="https://colab.research.google.com/github/TransformerLensOrg/TransformerLens/blob/main/demos/Boot_External_vLLM_Walkthrough.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# `boot_external` + vLLM: What Works, What Doesn't, What's Next

This is a *walkthrough*, not a copy-pasteable recipe. vLLM's internal model classes use fused/TP-aware modules that **do not match** the dot-paths the TransformerBridge architecture adapters expect, so wrapping a vLLM model directly with `boot_external` doesn't work today. This notebook lays out:

1. **Where the trees diverge** — concretely what vLLM does differently from HF.
2. **The state-dict transfer pattern** — works today, costs you 2× the memory.
3. **The vLLM-shaped adapter sketch** — the path to zero-overhead instrumentation, not yet implemented.
4. **A runnable mock** — pretends an HF Llama is "the vLLM model" so you can see the wiring without needing GPUs or a vLLM install.

**TL;DR**: if you need TL hooks active *during* vLLM generation today, option 2 is the realistic path and you'll pay the memory cost. The mid-term win — a vLLM adapter — is tracked as future work.

## Why the trees don't line up

Take an HF `LlamaDecoderLayer`. Its self-attention exposes four separate linear projections, and the `LlamaArchitectureAdapter` knows to walk to `model.layers.0.self_attn.q_proj`, `.k_proj`, `.v_proj`, `.o_proj`.

```
HF Llama:                                       vLLM Llama:
  model.layers[0].self_attn                       model.layers[0].self_attn
    .q_proj   (Linear)                              .qkv_proj  (QKVParallelLinear) ← fused
    .k_proj   (Linear)                              .o_proj    (RowParallelLinear)
    .v_proj   (Linear)                              .rotary_emb
    .o_proj   (Linear)                              .attn       (custom kernel)
    .rotary_emb
```

Two consequences:

1. **Path resolution fails.** `adapter.get_remote_component(vllm_model, "model.layers.0.self_attn.q_proj")` raises `AttributeError` — there *is* no `q_proj` on vLLM's `self_attn`. The bridge would crash during construction inside `set_original_components`.
2. **Even if you patched the paths,** the layouts wouldn't match. `qkv_proj.weight` is `[3*hidden, hidden]`, not three `[hidden, hidden]` tensors. Hook outputs would be a single fused tensor, not `q`/`k`/`v` separately. That's a real semantic difference, not just a renaming.

Beyond attention, vLLM also fuses MLP gate+up projections, uses TP-aware linears (`ColumnParallelLinear`, `RowParallelLinear`), and replaces RMSNorm with a custom CUDA op. None of those line up with the HF tree the adapters target.

## Pattern 1: state-dict transfer (works today)

vLLM's loaded model exposes a `state_dict` that *is* convertible to HF format — vLLM itself loads from HF checkpoints, so the mapping is well-defined (the inverse of vLLM's `_load_weights`). The pattern:

```
vLLM LLM(model="meta-llama/Llama-3.2-1B")         # 1× memory, 1× Hub call
      │
      ├── used for fast generation
      │
      └── state_dict ──→ unfuse_qkv ──→ HF LlamaForCausalLM (from_config + load_state_dict)
                                              │
                                              └── boot_external(...) ──→ TransformerBridge with hooks
                                                                          ↑
                                                  Now 2× memory — both copies live concurrently.
```

**Honest accounting:** you're holding two copies of the weights. This is fine for analysis-after-generation (extract state once, do interpretability separately) but it's not free. The memory doubling is the cost of vLLM's fused layout being incompatible with HF's tree.

The runnable cell below mocks this pattern — `pretend_vllm_load()` returns an HF Llama to stand in for vLLM's loaded model, since vLLM isn't installable in most demo environments.

## Setup

In [1]:
# NBVAL_IGNORE_OUTPUT
import os

IN_GITHUB = os.getenv("GITHUB_ACTIONS") == "true"
try:
    import google.colab

    IN_COLAB = True
    print("Running as a Colab notebook")
except ImportError:
    IN_COLAB = False

if not IN_GITHUB and not IN_COLAB:
    print("Running as a Jupyter notebook - intended for development only!")
    from IPython import get_ipython

    ipython = get_ipython()
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

if IN_COLAB or IN_GITHUB:
    %pip install transformer_lens

Running as a Jupyter notebook - intended for development only!


In [2]:
import torch
from transformers import AutoModelForCausalLM, LlamaConfig

from transformer_lens.model_bridge import TransformerBridge

### Mock vLLM load

In a real pipeline this would be:

```python
from vllm import LLM
llm = LLM(model="meta-llama/Llama-3.2-1B")
# vllm_internal = llm.llm_engine.model_executor.driver_worker.model_runner.model
# vllm_state_dict = vllm_internal.state_dict()
```

We can't run vLLM in this notebook environment, so we mock it with a tiny Llama. The point is to show what the *next* step looks like.

In [ ]:
def pretend_vllm_load():
    """Stand-in for a real vLLM load.

    In production this would return vLLM's internal model with its fused
    `qkv_proj` modules. Here we return a small HF Llama so the rest of the
    walkthrough is runnable — the *transfer pattern* is what we're
    demonstrating, not the vLLM kernel internals.
    """
    cfg = LlamaConfig(
        hidden_size=32,
        intermediate_size=64,
        num_hidden_layers=2,
        num_attention_heads=4,
        num_key_value_heads=4,
        vocab_size=128,
        max_position_embeddings=64,
    )
    return cfg, AutoModelForCausalLM.from_config(cfg)


vllm_cfg, vllm_model = pretend_vllm_load()
vllm_model.eval()
print("mock vLLM model loaded:", type(vllm_model).__name__)

### Transfer into an HF-shaped shell

For a *real* vLLM model, this step would need to unfuse the QKV weight. A sketch:

```python
def unfuse_qkv(vllm_state_dict, num_heads, num_kv_heads, head_dim):
    out = {}
    for k, v in vllm_state_dict.items():
        if k.endswith(".self_attn.qkv_proj.weight"):
            q_size = num_heads * head_dim
            kv_size = num_kv_heads * head_dim
            q, kk, vv = torch.split(v, [q_size, kv_size, kv_size], dim=0)
            base = k.removesuffix("qkv_proj.weight")
            out[base + "q_proj.weight"] = q
            out[base + "k_proj.weight"] = kk
            out[base + "v_proj.weight"] = vv
        elif k.endswith(".mlp.gate_up_proj.weight"):
            # vLLM also fuses gate+up — unfuse symmetrically
            mid = v.shape[0] // 2
            base = k.removesuffix("gate_up_proj.weight")
            out[base + "gate_proj.weight"] = v[:mid]
            out[base + "up_proj.weight"] = v[mid:]
        else:
            out[k] = v
    return out
```

In our mock, the model is already HF-shaped, so the transfer is a no-op `state_dict()` round-trip — but it makes the second copy concrete (note the memory doubling).

In [ ]:
# Real flow: vllm_state_dict = unfuse_qkv(vllm_internal.state_dict(), ...)
# Mock flow:
vllm_state_dict = vllm_model.state_dict()

# Build an HF-shaped shell — config matches what vLLM was loaded with.
hf_shell = AutoModelForCausalLM.from_config(vllm_cfg)
hf_shell.load_state_dict(vllm_state_dict)
hf_shell.eval()

print("weights transferred. Both `vllm_model` and `hf_shell` are now in memory.")
print(f"vllm_model params: {sum(p.numel() for p in vllm_model.parameters()):,}")
print(f"hf_shell   params: {sum(p.numel() for p in hf_shell.parameters()):,}")

### Wrap the shell with `boot_external`

Now we instrument the HF shell — not the vLLM model. The shell has the dot-paths the `LlamaArchitectureAdapter` expects, so `boot_external` wires up cleanly.

In [ ]:
bridge = TransformerBridge.boot_external(
    model=hf_shell,
    architecture="LlamaForCausalLM",
    hf_config=vllm_cfg,
    tokenizer=None,  # bring your own if needed
)
print(f"bridge wired: {len(bridge._hook_registry)} hooks")

report = TransformerBridge.diagnose_paths(
    model=hf_shell, architecture="LlamaForCausalLM", hf_config=vllm_cfg
)
print(f"resolved: {len(report['resolved'])}, missing: {len(report['missing'])}")

In [ ]:
# Run with cache to confirm hooks fire on real input.
tokens = torch.randint(0, vllm_cfg.vocab_size, (1, 8))
with torch.inference_mode():
    logits, cache = bridge.run_with_cache(tokens)

print(f"logits: {tuple(logits.shape)}")
print(f"cache:  {len(cache)} hooks captured")
print("first few:", sorted(cache.keys())[:5])

**That's the realistic vLLM-today pattern.** vLLM owns generation; the HF shell owns instrumentation. They share weights via state_dict transfer (one-time copy). You pay 2× memory; you keep vLLM's throughput on the generation side and TL's full hook surface on the analysis side.

## Pattern 2: a vLLM-shaped architecture adapter (future work)

The doubled memory is the price of bridging two tree shapes via state_dict transfer. The zero-overhead alternative is a **vLLM-shaped adapter** — a subclass of `LlamaArchitectureAdapter` that walks `qkv_proj` instead of separate `q_proj/k_proj/v_proj`, decodes the fused tensor into `q`/`k`/`v` hook outputs at runtime, and accounts for TP-aware linear layouts.

Sketch of what would need to change:

```python
class VLLMLlamaArchitectureAdapter(LlamaArchitectureAdapter):
    def __init__(self, cfg):
        super().__init__(cfg)
        # Replace the attention component mapping: one fused projection,
        # not three; the bridge synthesises q/k/v hooks from a forward-time
        # split rather than reading three submodules.
        self.component_mapping["blocks"].submodules["attn"] = JointQKVAttentionBridge(
            name="self_attn",
            qkv_path="qkv_proj",          # single fused module
            o_path="o_proj",
            qkv_split=(cfg.n_heads, cfg.n_key_value_heads, cfg.n_key_value_heads),
        )
        # Same idea for MLP — gate+up are fused as `gate_up_proj`.
        ...
```

There's already a `JointQKVAttentionBridge` in the codebase used for GPT-2's `c_attn` — the surface for fused attention exists. Wiring up the vLLM-side TP-aware linears and the custom RMSNorm op is the remaining work.

With a vLLM adapter in place, the call would look like:

```python
from vllm import LLM
llm = LLM(model="meta-llama/Llama-3.2-1B")
vllm_internal = llm.llm_engine.model_executor.driver_worker.model_runner.model

bridge = TransformerBridge.boot_external(
    model=vllm_internal,
    architecture="VLLMLlamaForCausalLM",   # new architecture key
    hf_config=llm.llm_engine.model_config.hf_config,
    tokenizer=llm.get_tokenizer(),
)
# Single copy of weights, hooks active during vLLM generation.
```

That's the eventual zero-overhead story. Until it ships, Pattern 1 is the working path.

## What `boot_external` actually adds here (and what it doesn't)

Worth being explicit about the overlap with the existing `boot_transformers(name, hf_model=...)` path:

**With the Pattern 1 state-dict-transfer flow shown above**, both entry points work equivalently. The HF shell is a `PreTrainedModel`, so you could replace the `boot_external(...)` call with:

```python
bridge = TransformerBridge.boot_transformers(
    "meta-llama/Llama-3.2-1B",  # any string; not used because hf_model is provided
    hf_model=hf_shell,
    tokenizer=tokenizer,
)
```

and get the same result. `boot_external` is *marginally* cleaner here — no required `model_name`, first-class `tokenizer=None`, no implicit `AutoTokenizer.from_pretrained` fallback — but the headline win (no second Hub download) belongs to *both* entry points, not specifically to `boot_external`.

**The case where `boot_external` is genuinely the only option** is Pattern 2 — a vLLM-shaped adapter that lets the bridge wrap vLLM's internal module *directly*, no state-dict transfer, no HF shell, no doubled memory. vLLM's internal model is **not** a `PreTrainedModel`, so `boot_transformers(hf_model=...)` would fail when it reaches for `hf_model.config`. `boot_external` accepts the config separately. That's the future-state value-add, and it's gated on the adapter work, not on the boot entry point.

**Bottom line**: `boot_external` doesn't unlock anything for vLLM *today* that `boot_transformers(hf_model=...)` doesn't also unlock. It clears the runway for the adapter-shaped solution that does.